# Week 8 Evaluation Expansion
- Compute metrics beyond R²: MAPE and MdAPE. (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.median_absolute_error.html)
    - Why is it difficult to find things on MdAPE? (https://stats.stackexchange.com/questions/596324/is-median-absolute-percentage-error-useless)
- Summarize insights (which price bands performed better)

- I'll clean up the code from previous files and re-do it here for better readability (and convenience)
- Hyperparameter tuning (https://www.geeksforgeeks.org/machine-learning/sklearn-model-hyper-parameters-tuning/)

In [ ]:
# Importing modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Hyperparameter tuning modules
from sklearn.model_selection import GridSearchCV

# Regression modules
from sklearn.linear_model import LinearRegression
from sklearn.metrics import median_absolute_error, mean_absolute_percentage_error

# Tree modules
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# XGBoost
from xgboost import XGBRegressor
from scipy.stats import loguniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

# Data Wrangling
Import and split data properly

In [52]:
# Importing cleaned data
root = "C:/Users/donutii/Desktop/IDX-Exchange-Internship-Data-Science-61"
data_location = f"{root}/IDX_Exchange/deliverables"

testing_set = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set.csv')
training_set = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set.csv') 

# feature engineered datasets
testing_set_fe = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set_fe.csv')
training_set_fe = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set_fe.csv') 

In [53]:
# Splitting training set into different months to test hyperparameter
months = ['2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2025-06']
training_set_split_by_months = []
for month in range(len(months)):
    training_set_split_by_months.append(training_set[training_set['CloseDate'].str.contains(months[month])])
    #(training_set_months[month].columns)
    training_set_split_by_months[month] = training_set_split_by_months[month].drop(columns='CloseDate')

# Get Target and Normal Vars from the Testing set
testing_vars = testing_set.drop(columns='ClosePrice')
testing_target = testing_set['ClosePrice']

In [54]:
# Do the same with featured engineered sets

training_set_months_fe = []
for month in range(len(months)):
    training_set_months_fe.append(training_set_fe[training_set_fe['CloseDate'].str.contains(months[month])])
    #(training_set_months[month].columns)
    training_set_months_fe[month] = training_set_months_fe[month].drop(columns='CloseDate')
    
testing_vars_fe = testing_set_fe.drop(columns='ClosePrice')
testing_target_fe = testing_set_fe['ClosePrice']

In [55]:
""" 
    Function to test different time frames of training data
    You can include up to 13 months
"""
def choose_test_months(num_months):
    if(num_months > 13 or num_months < 1):
        num_months = 13
        
    included_months = []
    for month in range(num_months):
        included_months.append(training_set_split_by_months[12 - month])
    
    return pd.concat(included_months, axis=0)

""" 
    Function to split trianing data into its variables and target
"""
def get_training_split(training_data):
    training_vars = training_data.drop(columns='ClosePrice')
    training_target = training_data['ClosePrice']
    
    return training_vars, training_target

In [56]:
# Creating dataframe used to store scores between models
evaluations = pd.DataFrame(columns=['Model', 'Training_R2', 'Training_MdAPE', 'Training_MAPE', 'R2', 'MdAPE', 'MAPE'])

# Baseline Model
- From 03_baseline_model.ipynb
- A simple regression model using sklearn
- Test out number of months as hyperparameter

In [68]:
def baseline_regression_model():
    # number of months
    num_months = [1, 6, 12, 13]
    
    regression_model_performance = pd.DataFrame(columns=['Num_Months', 'Training_R2', 'Training_MdAPE', 'Training_MAPE', 'R2', 'MdAPE', 'MAPE'])
    
    for num in num_months:
        training_data = choose_test_months(num_months=num)
        training_vars, training_target = get_training_split(training_data)
        
        # Create models
        model = LinearRegression().fit(training_vars, training_target)
        
        new_row = pd.DataFrame([{
            'Num_Months': num,
            'Training_R2': model.score(training_vars, training_target),
            'Training_MdAPE': median_absolute_error(training_target, model.predict(training_vars)),
            'Training_MAPE': mean_absolute_percentage_error(training_target, model.predict(training_vars)),
            'R2': model.score(testing_vars, testing_target),
            'MdAPE': median_absolute_error(testing_target, model.predict(testing_vars)),
            'MAPE': mean_absolute_percentage_error(testing_target, model.predict(testing_vars))
        }])

        regression_model_performance = pd.concat(
            [regression_model_performance, new_row],
            ignore_index=True
        )
        
        print(f"R2 of model trained on {num} months: {model.score(testing_vars, testing_target)}")

    
    regression_model_performance = regression_model_performance.sort_values(by='R2', ascending=False)
    print("Baseline Model Performance Summary:")    
    print(regression_model_performance)
    
    return regression_model_performance.iloc[0]
    
    
    

In [70]:
# Regress normal data
best_model = baseline_regression_model()

print("Baseline Regression Performance:")
print(f"Number of Months: {best_model['Num_Months']}")
print(f"Training R2: {best_model['Training_R2']}")
print(f"Training MdAPE: {best_model['Training_MdAPE']}")
print(f"Training MAPE: {best_model['Training_MAPE']}")
print(f"R2: {best_model['R2']}")
print(f"MdAPE: {best_model['MdAPE']}")
print(f"MAPE: {best_model['MAPE']}")

evaluations = pd.concat([evaluations, best_model], ignore_index=True)


C:\Users\donutii\AppData\Local\Temp\ipykernel_9116\1578255928.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  regression_model_performance = pd.concat(


R2 of model trained on 1 months: 0.4474074546268124
R2 of model trained on 6 months: 0.7018661811650542
R2 of model trained on 12 months: 0.7215168470647313
R2 of model trained on 13 months: 0.7215586516096621
Baseline Model Performance Summary:
  Num_Months  Training_R2  Training_MdAPE  Training_MAPE        R2  \
3         13     0.707523   190106.304610      13.891963  0.721559   
2         12     0.706574   189804.003664      13.893092  0.721517   
1          6     0.720403   185537.442318      14.667433  0.701866   
0          1     0.726964   184137.546210      13.865004  0.447407   

           MdAPE       MAPE  
3  189714.922489  12.060183  
2  189362.916179  12.031524  
1  184146.286611  12.381862  
0  422936.541098  22.311199  
Baseline Regression Performance:
Number of Months: 13
Training R2: 0.7075231518539578
Training MdAPE: 190106.30460956995
Training MAPE: 13.891962651288027
R2: 0.7215586516096621
MdAPE: 189714.92248880025
MAPE: 12.06018322331926


In [ ]:
# Regress feature engineered data

# Regression Decision Tree
- From 04_model_comparison.ipynb
- DecisionTree using sklearn
- DecisionTrees are also relatively cheap in processing so I will use grid search here as well

In [72]:
def decisiontree_regression_model(training_vars, training_target):
    params = {
        "criterion" : ['squared_error', 'friedman_mse', 'poisson'],
        "depths" : [20, 30, 40, 60]
    }
    
    # Create a GridSearch CV
    model_grid_searchcv = GridSearchCV(DecisionTreeRegressor(), params, cv=5, scoring=['r2', 'neg_median_absolute_error', 'neg_mean_absolute_percentage_error'], n_jobs=-1)
    
    # Fit to training data
    model_grid_searchcv.fit(training_vars, training_target)
    
    # create cv
    columns = [f"param_{name}" for name in params.keys()]
    columns += ["r2", "MdAPE", 'MAPE']
    cv_results = pd.DataFrame(model_grid_searchcv.cv_results_)
    cv_results["r2"] = -cv_results["r2"]
    cv_results["MdAPE"] = -cv_results["neg_median_absolute_error"]
    cv_results["MAPE"] = cv_results["neg_mean_absolute_percentage_error"]
    cv_results[columns].sort_values(by="r2")
    
    print(cv_results)
    
    # return the best model
    return model_grid_searchcv.best_estimator_

In [73]:
# regress normal data

num_months = [1, 6, 12, 13]
for months in num_months:
    training_data = choose_test_months(num_months=months)
    training_vars, training_target = get_training_split(training_data)
    
    best_model = decisiontree_regression_model(training_vars, training_target)
    
    new_row = pd.DataFrame([{
        'Model': f'DecisionTreeRegressor_{months}months',
        'Training_R2': best_model.score(training_vars, training_target),
        'Training_MdAPE': median_absolute_error(training_target, best_model.predict(training_vars)),
        'Training_MAPE': mean_absolute_percentage_error(training_target, best_model.predict(training_vars)),
        'R2': best_model.score(testing_vars, testing_target),
        'MdAPE': median_absolute_error(testing_target, best_model.predict(testing_vars)),
        'MAPE': mean_absolute_percentage_error(testing_target, best_model.predict(testing_vars))
    }])
    
    evaluations = pd.concat([evaluations, new_row], ignore_index=True)

ValueError: For multi-metric scoring, the parameter refit must be set to a scorer key or a callable to refit an estimator with the best parameter setting on the whole data and make the best_* attributes available for that metric. If this is not needed, refit should be set to False explicitly. True was passed.

# Random Forest Regressor
- Also from 04_model_comparison.ipynb
- Using randomsearch because processing these takes quite a while

In [ ]:
def randomforest_regression_model(training_vars, training_target):
    params = {
        "criterion" : ['squared_error', 'friedman_mse', 'poisson'],
        "depths" : [20, 30, 40, 60],
        # TODO: get more params here
    }
    
    # Create a Random Search CV
    model_rand_searchcv = RandomizedSearchCV(RandomForestRegressor(), params, cv=5, scoring=['r2', 'neg_median_absolute_error', 'neg_mean_absolute_percentage_error'], n_jobs=-1)
    
    # Fit to training data
    model_rand_searchcv.fit(training_vars, training_target)
    
    # create cv
    columns = [f"param_{name}" for name in params.keys()]
    columns += ["r2", "MdAPE", 'MAPE']
    cv_results = pd.DataFrame(model_rand_searchcv.cv_results_)
    cv_results["r2"] = -cv_results["r2"]
    cv_results["MdAPE"] = -cv_results["neg_median_absolute_error"]
    cv_results["MAPE"] = cv_results["neg_mean_absolute_percentage_error"]
    cv_results[columns].sort_values(by="r2")
    
    # return the best model
    return model_rand_searchcv.best_estimator_

In [ ]:
# regress normal data

# XGBoost
- From 05_advanced_models.ipynb
- 

In [ ]:
def xgboost_regression_model(training_vars, training_target):
    params = {
        "max_iter": [100, 300, 500, 600, 1000],
        "max_leaf_nodes": [5, 10, 20, 50, 75, 100],
        "learning_rate": loguniform(0.01, 1),
    }
    
    # Create a GridSearch CV
    model_grid_searchcv = RandomizedSearchCV(HistGradientBoostingRegressor(), params, cv=5, scoring=['r2', 'neg_median_absolute_error', 'neg_mean_absolute_percentage_error'], n_jobs=-1)
    
    # Fit to training data
    model_grid_searchcv.fit(training_vars, training_target)
    
    # create cv
    columns = [f"param_{name}" for name in params.keys()]
    columns += ["r2", "MdAPE", 'MAPE']
    cv_results = pd.DataFrame(model_grid_searchcv.cv_results_)
    cv_results["r2"] = -cv_results["r2"]
    cv_results["MdAPE"] = -cv_results["neg_median_absolute_error"]
    cv_results["MAPE"] = cv_results["neg_mean_absolute_percentage_error"]
    cv_results[columns].sort_values(by="r2")
    
    # return the best model
    return model_grid_searchcv.best_estimator_

In [ ]:
# Regress normal data

# Consolidate evaluations
- Ideally add the best of each model to a csv and then export it

In [ ]:
# Consolidate and export evaluations here